> Note: This notebook is an exercise workbook companion. Some cells intentionally contain `TODO` prompts or `NotImplementedError` placeholders for the reader to complete. For fully maintained runnable reference code, see `src/`, `tests/`, and the printed listings in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-repo/Optimization-for-Machine-Learning/blob/main/notebooks/ch05_loss_functions.ipynb)

# Chapter 5: Loss Functions

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Understand** the mathematical foundations of common loss functions
2. **Implement** cross-entropy, MSE, MAE, and other loss functions from scratch
3. **Visualize** loss landscapes and understand their properties
4. **Compare** different loss functions for various tasks
5. **Create** custom loss functions for specific problems
6. **Analyze** gradient flow through different losses

---

## Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import LogNorm
from mpl_toolkits.mplot3d import Axes3D
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

# For reproducibility
np.random.seed(42)

print("Setup complete!")

## 1. Introduction to Loss Functions

A **loss function** (also called cost function or objective function) measures how well our model's predictions match the true values. The goal of training is to minimize this loss.

### Properties of Good Loss Functions

1. **Differentiable**: We need gradients for optimization
2. **Convex** (ideally): Guarantees global optimum
3. **Meaningful**: Should relate to the task's evaluation metric
4. **Computationally efficient**: Fast to compute for large datasets

### Common Loss Functions

| Task | Loss Function | Formula |
|------|--------------|--------|
| Regression | MSE | $\frac{1}{n}\sum(y - \hat{y})^2$ |
| Regression | MAE | $\frac{1}{n}\sum|y - \hat{y}|$ |
| Regression | Huber | Combination of MSE and MAE |
| Binary Classification | Binary Cross-Entropy | $-[y\log(\hat{y}) + (1-y)\log(1-\hat{y})]$ |
| Multi-class Classification | Categorical Cross-Entropy | $-\sum y_i \log(\hat{y}_i)$ |

## 2. Cross-Entropy Loss Explained

Cross-entropy loss is the most common loss function for classification tasks. Let's understand it from first principles.

### Information Theory Background

**Entropy** measures the uncertainty in a probability distribution:
$$H(p) = -\sum_i p_i \log(p_i)$$

**Cross-Entropy** measures the difference between two distributions:
$$H(p, q) = -\sum_i p_i \log(q_i)$$

For classification:
- $p$ is the true distribution (one-hot encoded)
- $q$ is the predicted distribution (softmax outputs)

### Binary Cross-Entropy

For binary classification with true label $y \in \{0, 1\}$ and predicted probability $\hat{y} \in [0, 1]$:

$$\mathcal{L} = -[y \log(\hat{y}) + (1-y) \log(1 - \hat{y})]$$

In [ ]:
# Implement Cross-Entropy Loss from scratch

class BinaryCrossEntropy:
    """Binary Cross-Entropy Loss.
    
    L = -[y * log(y_hat) + (1-y) * log(1 - y_hat)]
    
    Used for binary classification problems.
    """
    
    def __init__(self, epsilon=1e-15):
        """Initialize with small epsilon for numerical stability."""
        self.epsilon = epsilon
        self.cache = {}
    
    def forward(self, y_pred, y_true):
        """Compute the binary cross-entropy loss.
        
        Parameters:
        -----------
        y_pred : numpy.ndarray
            Predicted probabilities, shape (n_samples,) or (n_samples, 1)
        y_true : numpy.ndarray
            True labels (0 or 1), same shape as y_pred
            
        Returns:
        --------
        float
            Average loss over all samples
        """
        # Clip predictions to avoid log(0)
        y_pred = np.clip(y_pred, self.epsilon, 1 - self.epsilon)
        
        # Store for backward pass
        self.cache['y_pred'] = y_pred
        self.cache['y_true'] = y_true
        
        # Compute loss: -[y*log(p) + (1-y)*log(1-p)]
        loss = -(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
        return np.mean(loss)
    
    def backward(self):
        """Compute gradient of loss with respect to predictions.
        
        dL/dy_pred = -y/y_pred + (1-y)/(1-y_pred)
                   = (y_pred - y) / (y_pred * (1 - y_pred))
        """
        y_pred = self.cache['y_pred']
        y_true = self.cache['y_true']
        n_samples = y_true.shape[0]
        
        # Gradient: (y_pred - y_true) / (y_pred * (1 - y_pred))
        grad = (y_pred - y_true) / (y_pred * (1 - y_pred) + self.epsilon)
        return grad / n_samples


class CategoricalCrossEntropy:
    """Categorical Cross-Entropy Loss.
    
    L = -sum(y_true * log(y_pred))
    
    Used for multi-class classification with softmax outputs.
    """
    
    def __init__(self, epsilon=1e-15):
        self.epsilon = epsilon
        self.cache = {}
    
    def forward(self, y_pred, y_true):
        """Compute categorical cross-entropy loss.
        
        Parameters:
        -----------
        y_pred : numpy.ndarray
            Predicted probabilities, shape (n_samples, n_classes)
        y_true : numpy.ndarray
            One-hot encoded true labels, shape (n_samples, n_classes)
            
        Returns:
        --------
        float
            Average loss over all samples
        """
        # Clip predictions
        y_pred = np.clip(y_pred, self.epsilon, 1 - self.epsilon)
        
        self.cache['y_pred'] = y_pred
        self.cache['y_true'] = y_true
        
        # Compute loss: -sum(y_true * log(y_pred))
        loss = -np.sum(y_true * np.log(y_pred), axis=1)
        return np.mean(loss)
    
    def backward(self):
        """Compute gradient of loss with respect to predictions.
        
        dL/dy_pred = -y_true / y_pred
        
        Note: When combined with softmax, the gradient simplifies to (y_pred - y_true)
        """
        y_pred = self.cache['y_pred']
        y_true = self.cache['y_true']
        n_samples = y_true.shape[0]
        
        grad = -y_true / (y_pred + self.epsilon)
        return grad / n_samples

print("Cross-entropy loss classes defined!")

In [ ]:
# Visualize Binary Cross-Entropy Loss

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Loss vs predicted probability for different true labels
ax = axes[0]
y_pred_range = np.linspace(0.001, 0.999, 200)

bce = BinaryCrossEntropy()

# When true label is 1
loss_y1 = [bce.forward(np.array([p]), np.array([1])) for p in y_pred_range]
ax.plot(y_pred_range, loss_y1, 'b-', linewidth=2, label='True label = 1')

# When true label is 0
loss_y0 = [bce.forward(np.array([p]), np.array([0])) for p in y_pred_range]
ax.plot(y_pred_range, loss_y0, 'r-', linewidth=2, label='True label = 0')

ax.set_xlabel('Predicted Probability', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Binary Cross-Entropy Loss', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 5)

# Plot 2: Gradient of BCE
ax = axes[1]

grad_y1 = []
grad_y0 = []
for p in y_pred_range:
    bce.forward(np.array([p]), np.array([1]))
    grad_y1.append(bce.backward()[0])
    bce.forward(np.array([p]), np.array([0]))
    grad_y0.append(bce.backward()[0])

ax.plot(y_pred_range, grad_y1, 'b-', linewidth=2, label='True label = 1')
ax.plot(y_pred_range, grad_y0, 'r-', linewidth=2, label='True label = 0')
ax.axhline(y=0, color='black', linestyle='--', linewidth=1)

ax.set_xlabel('Predicted Probability', fontsize=12)
ax.set_ylabel('Gradient', fontsize=12)
ax.set_title('Gradient of BCE\n(Larger gradients for wrong predictions)', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(-10, 10)

# Plot 3: 2D loss surface for two classes
ax = axes[2]
p1 = np.linspace(0.01, 0.99, 100)
p2 = np.linspace(0.01, 0.99, 100)
P1, P2 = np.meshgrid(p1, p2)

# Loss for true label [1, 0] (class 0)
loss_surface = -(np.log(P1) + 0)  # Only first class matters for this true label

contour = ax.contourf(P1, P2, loss_surface, levels=30, cmap='viridis')
plt.colorbar(contour, ax=ax, label='Loss')
ax.plot(1, 0, 'r*', markersize=20, label='Optimal prediction')

ax.set_xlabel('P(class 0)', fontsize=12)
ax.set_ylabel('P(class 1)', fontsize=12)
ax.set_title('Cross-Entropy Surface\n(True label: class 0)', fontsize=14)
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Demonstrate why cross-entropy is preferred over MSE for classification

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

# Compare gradients for BCE vs MSE when predictions are very wrong
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

z_range = np.linspace(-6, 6, 200)
y_true = 1  # True label is 1

# For BCE: dL/dz = sigmoid(z) - y_true (when combined with sigmoid)
bce_grad = sigmoid(z_range) - y_true

# For MSE: dL/dz = 2 * (sigmoid(z) - y_true) * sigmoid'(z)
mse_grad = 2 * (sigmoid(z_range) - y_true) * sigmoid_derivative(z_range)

# Plot gradients
ax = axes[0]
ax.plot(z_range, np.abs(bce_grad), 'b-', linewidth=2, label='BCE gradient magnitude')
ax.plot(z_range, np.abs(mse_grad), 'r-', linewidth=2, label='MSE gradient magnitude')
ax.axvline(x=0, color='green', linestyle='--', label='Decision boundary')

ax.set_xlabel('Logit (z)', fontsize=12)
ax.set_ylabel('|Gradient|', fontsize=12)
ax.set_title('Gradient Magnitude: BCE vs MSE\n(True label = 1)', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Annotate key regions
ax.annotate('Very wrong\n(large BCE grad)', xy=(-4, 0.9), fontsize=10,
            bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))
ax.annotate('Very wrong\n(small MSE grad!)', xy=(-4, 0.05), fontsize=10,
            bbox=dict(boxstyle='round', facecolor='red', alpha=0.3))

# Plot predictions
ax = axes[1]
ax.plot(z_range, sigmoid(z_range), 'purple', linewidth=2, label='Prediction (sigmoid(z))')
ax.axhline(y=1, color='green', linestyle='--', label='True label = 1')
ax.axvline(x=0, color='gray', linestyle=':', alpha=0.5)

ax.set_xlabel('Logit (z)', fontsize=12)
ax.set_ylabel('Probability', fontsize=12)
ax.set_title('Sigmoid Predictions', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey Insight:")
print("When z << 0 (very wrong prediction for y=1), BCE gradient is large,")
print("but MSE gradient is small due to sigmoid saturation.")
print("This is why BCE leads to faster and more stable training for classification!")

## 3. MSE vs MAE Comparison

For regression tasks, the two most common loss functions are:

**Mean Squared Error (MSE)**:
$$\mathcal{L}_{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

**Mean Absolute Error (MAE)**:
$$\mathcal{L}_{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

In [ ]:
# Implement MSE and MAE from scratch

class MeanSquaredError:
    """Mean Squared Error Loss.
    
    L = (1/n) * sum((y - y_hat)^2)
    
    Properties:
    - Differentiable everywhere
    - Penalizes large errors heavily (quadratic)
    - Sensitive to outliers
    - Optimal for Gaussian noise
    """
    
    def __init__(self):
        self.cache = {}
    
    def forward(self, y_pred, y_true):
        """Compute MSE loss."""
        self.cache['y_pred'] = y_pred
        self.cache['y_true'] = y_true
        
        return np.mean((y_true - y_pred) ** 2)
    
    def backward(self):
        """Compute gradient: dL/dy_pred = -2(y_true - y_pred) / n = 2(y_pred - y_true) / n"""
        y_pred = self.cache['y_pred']
        y_true = self.cache['y_true']
        n = y_true.shape[0]
        
        return 2 * (y_pred - y_true) / n


class MeanAbsoluteError:
    """Mean Absolute Error Loss.
    
    L = (1/n) * sum(|y - y_hat|)
    
    Properties:
    - Not differentiable at 0 (use subgradient)
    - Penalizes all errors equally (linear)
    - Robust to outliers
    - Optimal for Laplace noise
    """
    
    def __init__(self):
        self.cache = {}
    
    def forward(self, y_pred, y_true):
        """Compute MAE loss."""
        self.cache['y_pred'] = y_pred
        self.cache['y_true'] = y_true
        
        return np.mean(np.abs(y_true - y_pred))
    
    def backward(self):
        """Compute gradient: dL/dy_pred = sign(y_pred - y_true) / n"""
        y_pred = self.cache['y_pred']
        y_true = self.cache['y_true']
        n = y_true.shape[0]
        
        # Sign function for gradient
        return np.sign(y_pred - y_true) / n


class HuberLoss:
    """Huber Loss (Smooth L1).
    
    L = 0.5 * (y - y_hat)^2           if |y - y_hat| <= delta
    L = delta * |y - y_hat| - 0.5 * delta^2  otherwise
    
    Combines MSE for small errors and MAE for large errors.
    """
    
    def __init__(self, delta=1.0):
        self.delta = delta
        self.cache = {}
    
    def forward(self, y_pred, y_true):
        """Compute Huber loss."""
        self.cache['y_pred'] = y_pred
        self.cache['y_true'] = y_true
        
        error = y_true - y_pred
        abs_error = np.abs(error)
        
        # Piecewise loss
        quadratic = 0.5 * error ** 2
        linear = self.delta * abs_error - 0.5 * self.delta ** 2
        
        loss = np.where(abs_error <= self.delta, quadratic, linear)
        return np.mean(loss)
    
    def backward(self):
        """Compute gradient."""
        y_pred = self.cache['y_pred']
        y_true = self.cache['y_true']
        n = y_true.shape[0]
        
        error = y_pred - y_true
        abs_error = np.abs(error)
        
        # Piecewise gradient
        grad = np.where(abs_error <= self.delta, 
                        error,  # MSE gradient
                        self.delta * np.sign(error))  # MAE gradient (scaled)
        return grad / n

print("MSE, MAE, and Huber loss classes defined!")

In [ ]:
# Visualize MSE vs MAE vs Huber

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

error_range = np.linspace(-5, 5, 200)

# Instantiate losses
mse = MeanSquaredError()
mae = MeanAbsoluteError()
huber = HuberLoss(delta=1.0)

# Compute losses for different errors
mse_losses = [mse.forward(np.array([e]), np.array([0])) for e in error_range]
mae_losses = [mae.forward(np.array([e]), np.array([0])) for e in error_range]
huber_losses = [huber.forward(np.array([e]), np.array([0])) for e in error_range]

# Plot 1: Loss values
ax = axes[0, 0]
ax.plot(error_range, mse_losses, 'b-', linewidth=2, label='MSE')
ax.plot(error_range, mae_losses, 'r-', linewidth=2, label='MAE')
ax.plot(error_range, huber_losses, 'g-', linewidth=2, label='Huber (delta=1)')

ax.set_xlabel('Error (y_pred - y_true)', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Loss Functions Comparison', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 10)

# Plot 2: Gradients
ax = axes[0, 1]

mse_grads = []
mae_grads = []
huber_grads = []

for e in error_range:
    mse.forward(np.array([e]), np.array([0]))
    mse_grads.append(mse.backward()[0])
    
    mae.forward(np.array([e]), np.array([0]))
    mae_grads.append(mae.backward()[0])
    
    huber.forward(np.array([e]), np.array([0]))
    huber_grads.append(huber.backward()[0])

ax.plot(error_range, mse_grads, 'b-', linewidth=2, label='MSE gradient')
ax.plot(error_range, mae_grads, 'r-', linewidth=2, label='MAE gradient')
ax.plot(error_range, huber_grads, 'g-', linewidth=2, label='Huber gradient')
ax.axhline(y=0, color='black', linestyle='--', linewidth=1)

ax.set_xlabel('Error (y_pred - y_true)', fontsize=12)
ax.set_ylabel('Gradient', fontsize=12)
ax.set_title('Gradient Comparison', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Plot 3: Effect of outliers
ax = axes[1, 0]

# Data with and without outlier
y_true_clean = np.array([1, 2, 3, 4, 5])
y_true_outlier = np.array([1, 2, 3, 4, 50])  # Outlier at position 4

y_pred_range = np.linspace(0, 15, 100)

# MSE with outlier vs clean
mse_clean = [np.mean((y_true_clean - p)**2) for p in y_pred_range]
mse_outlier = [np.mean((y_true_outlier - p)**2) for p in y_pred_range]

mae_clean = [np.mean(np.abs(y_true_clean - p)) for p in y_pred_range]
mae_outlier = [np.mean(np.abs(y_true_outlier - p)) for p in y_pred_range]

ax.plot(y_pred_range, mse_clean, 'b-', linewidth=2, label='MSE (clean)')
ax.plot(y_pred_range, mse_outlier, 'b--', linewidth=2, label='MSE (with outlier)')
ax.plot(y_pred_range, mae_clean, 'r-', linewidth=2, label='MAE (clean)')
ax.plot(y_pred_range, mae_outlier, 'r--', linewidth=2, label='MAE (with outlier)')

# Mark optimal predictions
ax.axvline(x=np.mean(y_true_clean), color='blue', linestyle=':', alpha=0.5)
ax.axvline(x=np.median(y_true_outlier), color='red', linestyle=':', alpha=0.5)

ax.set_xlabel('Constant Prediction', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Outlier Robustness\n(Data: [1,2,3,4,5] vs [1,2,3,4,50])', fontsize=14)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 500)

# Plot 4: Optimal prediction under each loss
ax = axes[1, 1]

# Generate data with outliers
np.random.seed(42)
y_data = np.concatenate([np.random.normal(5, 1, 95), np.array([20, 25, 30, 35, 40])])  # 5 outliers

ax.hist(y_data, bins=30, alpha=0.5, color='gray', edgecolor='black')
ax.axvline(x=np.mean(y_data), color='blue', linewidth=2, linestyle='-', label=f'Mean: {np.mean(y_data):.2f} (MSE optimal)')
ax.axvline(x=np.median(y_data), color='red', linewidth=2, linestyle='-', label=f'Median: {np.median(y_data):.2f} (MAE optimal)')

ax.set_xlabel('Value', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Optimal Constant Predictions\n(Data with outliers)', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey Insights:")
print("1. MSE penalizes large errors heavily -> sensitive to outliers")
print("2. MAE penalizes all errors equally -> robust to outliers")
print("3. MSE optimal prediction is the mean, MAE optimal is the median")
print("4. Huber loss combines benefits of both!")

## 4. Loss Landscape Gallery

Let's visualize the loss landscapes for different loss functions to understand their optimization properties.

In [ ]:
# Create 3D loss landscape visualizations

def create_loss_landscape(loss_fn, title, ax, y_true=1.0):
    """Create a 3D visualization of the loss landscape.
    
    For simplicity, we consider a 2D weight space where
    y_pred = w1 * x1 + w2 * x2 with fixed x1=1, x2=1
    """
    w1_range = np.linspace(-2, 2, 100)
    w2_range = np.linspace(-2, 2, 100)
    W1, W2 = np.meshgrid(w1_range, w2_range)
    
    # y_pred = w1 * 1 + w2 * 1 = w1 + w2
    Y_pred = W1 + W2
    
    # Compute loss at each point
    Loss = np.zeros_like(W1)
    for i in range(W1.shape[0]):
        for j in range(W1.shape[1]):
            Loss[i, j] = loss_fn(np.array([Y_pred[i, j]]), np.array([y_true]))
    
    # 3D surface
    surf = ax.plot_surface(W1, W2, Loss, cmap='viridis', alpha=0.8,
                           linewidth=0, antialiased=True)
    
    # Mark the optimal point (where w1 + w2 = y_true)
    ax.plot([y_true/2], [y_true/2], [0], 'r*', markersize=15)
    
    ax.set_xlabel('$w_1$')
    ax.set_ylabel('$w_2$')
    ax.set_zlabel('Loss')
    ax.set_title(title)
    
    return surf

# Create figure with 3D subplots
fig = plt.figure(figsize=(18, 12))

# MSE
ax1 = fig.add_subplot(2, 3, 1, projection='3d')
mse = MeanSquaredError()
create_loss_landscape(mse.forward, 'MSE Loss Landscape', ax1)

# MAE
ax2 = fig.add_subplot(2, 3, 2, projection='3d')
mae = MeanAbsoluteError()
create_loss_landscape(mae.forward, 'MAE Loss Landscape', ax2)

# Huber
ax3 = fig.add_subplot(2, 3, 3, projection='3d')
huber = HuberLoss(delta=0.5)
create_loss_landscape(huber.forward, 'Huber Loss Landscape', ax3)

# Contour plots (2D view)
def create_loss_contour(loss_fn, title, ax, y_true=1.0):
    w1_range = np.linspace(-2, 2, 100)
    w2_range = np.linspace(-2, 2, 100)
    W1, W2 = np.meshgrid(w1_range, w2_range)
    Y_pred = W1 + W2
    
    Loss = np.zeros_like(W1)
    for i in range(W1.shape[0]):
        for j in range(W1.shape[1]):
            Loss[i, j] = loss_fn(np.array([Y_pred[i, j]]), np.array([y_true]))
    
    contour = ax.contour(W1, W2, Loss, levels=20, cmap='viridis')
    ax.contourf(W1, W2, Loss, levels=20, cmap='viridis', alpha=0.3)
    
    # Mark optimal line (w1 + w2 = y_true)
    w1_opt = np.linspace(-2, 2, 100)
    w2_opt = y_true - w1_opt
    ax.plot(w1_opt, w2_opt, 'r--', linewidth=2, label='Optimal solutions')
    
    ax.set_xlabel('$w_1$')
    ax.set_ylabel('$w_2$')
    ax.set_title(title)
    ax.legend()
    ax.set_aspect('equal')

ax4 = fig.add_subplot(2, 3, 4)
create_loss_contour(mse.forward, 'MSE Contours', ax4)

ax5 = fig.add_subplot(2, 3, 5)
create_loss_contour(mae.forward, 'MAE Contours', ax5)

ax6 = fig.add_subplot(2, 3, 6)
create_loss_contour(huber.forward, 'Huber Contours', ax6)

plt.tight_layout()
plt.show()

In [ ]:
# More complex loss landscape: cross-entropy for binary classification

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Parameters: w1 and w2 for a simple linear classifier
# y_pred = sigmoid(w1 * x1 + w2 * x2)

# Generate a simple dataset
np.random.seed(42)
X_simple = np.array([[1, 0], [0, 1], [1, 1], [0, 0]])
y_simple = np.array([1, 1, 1, 0])  # XOR-like but separable

def compute_bce_loss(w1, w2, X, y):
    """Compute BCE loss for given weights."""
    z = w1 * X[:, 0] + w2 * X[:, 1]
    y_pred = sigmoid(z)
    y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
    loss = -(y * np.log(y_pred) + (1 - y) * np.log(1 - y_pred))
    return np.mean(loss)

# Create loss landscape
w1_range = np.linspace(-5, 5, 100)
w2_range = np.linspace(-5, 5, 100)
W1, W2 = np.meshgrid(w1_range, w2_range)

Loss = np.zeros_like(W1)
for i in range(W1.shape[0]):
    for j in range(W1.shape[1]):
        Loss[i, j] = compute_bce_loss(W1[i, j], W2[i, j], X_simple, y_simple)

# 3D plot
ax = fig.add_subplot(1, 3, 1, projection='3d')
surf = ax.plot_surface(W1, W2, Loss, cmap='viridis', alpha=0.8)
ax.set_xlabel('$w_1$')
ax.set_ylabel('$w_2$')
ax.set_zlabel('Loss')
ax.set_title('BCE Loss Landscape')

# Contour plot
ax = axes[1]
contour = ax.contour(W1, W2, Loss, levels=30, cmap='viridis')
ax.contourf(W1, W2, Loss, levels=30, cmap='viridis', alpha=0.3)
plt.colorbar(contour, ax=ax)

# Find and mark minimum
min_idx = np.unravel_index(np.argmin(Loss), Loss.shape)
ax.plot(W1[min_idx], W2[min_idx], 'r*', markersize=15, label=f'Min at ({W1[min_idx]:.2f}, {W2[min_idx]:.2f})')

ax.set_xlabel('$w_1$')
ax.set_ylabel('$w_2$')
ax.set_title('BCE Contours')
ax.legend()

# Decision boundaries at different points
ax = axes[2]
ax.scatter(X_simple[y_simple==1, 0], X_simple[y_simple==1, 1], 
           c='blue', s=100, label='Class 1', marker='o')
ax.scatter(X_simple[y_simple==0, 0], X_simple[y_simple==0, 1], 
           c='red', s=100, label='Class 0', marker='x')

# Draw decision boundary for optimal weights
x_boundary = np.linspace(-0.5, 1.5, 100)
# Decision boundary: w1*x1 + w2*x2 = 0
if abs(W2[min_idx]) > 0.01:
    y_boundary = -W1[min_idx] * x_boundary / W2[min_idx]
    ax.plot(x_boundary, y_boundary, 'g-', linewidth=2, label='Decision boundary')

ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('Data and Decision Boundary')
ax.legend()
ax.set_xlim(-0.5, 1.5)
ax.set_ylim(-0.5, 1.5)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Custom Loss Function Creation

Sometimes standard loss functions don't capture what we want to optimize. Let's learn how to create custom loss functions.

In [ ]:
# Example 1: Asymmetric Loss (penalize underpredictions more than overpredictions)

class AsymmetricLoss:
    """Asymmetric Loss that penalizes errors differently based on sign.
    
    Useful when:
    - Underpredicting is worse than overpredicting (or vice versa)
    - Examples: demand forecasting, risk estimation
    
    L = alpha * (y - y_hat)^2  if y > y_hat (underprediction)
    L = (1-alpha) * (y - y_hat)^2  if y <= y_hat (overprediction)
    """
    
    def __init__(self, alpha=0.7):
        """Initialize with asymmetry parameter.
        
        alpha > 0.5: penalize underpredictions more
        alpha < 0.5: penalize overpredictions more
        """
        self.alpha = alpha
        self.cache = {}
    
    def forward(self, y_pred, y_true):
        self.cache['y_pred'] = y_pred
        self.cache['y_true'] = y_true
        
        error = y_true - y_pred
        
        # Asymmetric weighting
        weights = np.where(error > 0, self.alpha, 1 - self.alpha)
        loss = weights * error ** 2
        
        return np.mean(loss)
    
    def backward(self):
        y_pred = self.cache['y_pred']
        y_true = self.cache['y_true']
        n = y_true.shape[0]
        
        error = y_true - y_pred
        weights = np.where(error > 0, self.alpha, 1 - self.alpha)
        
        grad = -2 * weights * error / n
        return grad


# Example 2: Quantile Loss

class QuantileLoss:
    """Quantile Loss for predicting specific quantiles.
    
    L = q * max(y - y_hat, 0) + (1 - q) * max(y_hat - y, 0)
    
    Useful for:
    - Predicting confidence intervals
    - Risk assessment (e.g., VaR)
    
    q = 0.5 gives MAE (median)
    q = 0.9 gives 90th percentile
    """
    
    def __init__(self, quantile=0.5):
        self.q = quantile
        self.cache = {}
    
    def forward(self, y_pred, y_true):
        self.cache['y_pred'] = y_pred
        self.cache['y_true'] = y_true
        
        error = y_true - y_pred
        loss = np.where(error >= 0,
                        self.q * error,
                        (self.q - 1) * error)
        
        return np.mean(loss)
    
    def backward(self):
        y_pred = self.cache['y_pred']
        y_true = self.cache['y_true']
        n = y_true.shape[0]
        
        error = y_true - y_pred
        grad = np.where(error >= 0, -self.q, 1 - self.q)
        
        return grad / n


# Example 3: Focal Loss (for imbalanced classification)

class FocalLoss:
    """Focal Loss for handling class imbalance.
    
    L = -alpha * (1 - p)^gamma * log(p)  for positive class
    L = -(1-alpha) * p^gamma * log(1-p)  for negative class
    
    gamma: focusing parameter (higher = more focus on hard examples)
    alpha: class weighting
    
    From "Focal Loss for Dense Object Detection" (Lin et al., 2017)
    """
    
    def __init__(self, gamma=2.0, alpha=0.25):
        self.gamma = gamma
        self.alpha = alpha
        self.epsilon = 1e-15
        self.cache = {}
    
    def forward(self, y_pred, y_true):
        y_pred = np.clip(y_pred, self.epsilon, 1 - self.epsilon)
        
        self.cache['y_pred'] = y_pred
        self.cache['y_true'] = y_true
        
        # Focal weights
        pt = np.where(y_true == 1, y_pred, 1 - y_pred)
        alpha_t = np.where(y_true == 1, self.alpha, 1 - self.alpha)
        
        focal_weight = alpha_t * (1 - pt) ** self.gamma
        
        # Cross entropy
        ce = -np.where(y_true == 1, np.log(y_pred), np.log(1 - y_pred))
        
        loss = focal_weight * ce
        return np.mean(loss)
    
    def backward(self):
        y_pred = self.cache['y_pred']
        y_true = self.cache['y_true']
        n = y_true.shape[0]
        
        pt = np.where(y_true == 1, y_pred, 1 - y_pred)
        alpha_t = np.where(y_true == 1, self.alpha, 1 - self.alpha)
        
        # Gradient computation (simplified)
        grad = alpha_t * (1 - pt) ** self.gamma * (self.gamma * pt * np.log(pt + self.epsilon) + pt - 1)
        grad = np.where(y_true == 1, -grad / (y_pred + self.epsilon), grad / (1 - y_pred + self.epsilon))
        
        return grad / n

print("Custom loss functions defined!")

In [ ]:
# Visualize custom loss functions

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Asymmetric Loss
ax = axes[0, 0]
error_range = np.linspace(-3, 3, 200)

for alpha in [0.3, 0.5, 0.7, 0.9]:
    asym = AsymmetricLoss(alpha=alpha)
    losses = [asym.forward(np.array([e]), np.array([0])) for e in error_range]
    ax.plot(error_range, losses, linewidth=2, label=f'alpha={alpha}')

ax.axvline(x=0, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('Error (y_pred - y_true)', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Asymmetric Loss\n(higher alpha = penalize underprediction more)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# 2. Quantile Loss
ax = axes[0, 1]

for q in [0.1, 0.25, 0.5, 0.75, 0.9]:
    quant = QuantileLoss(quantile=q)
    losses = [quant.forward(np.array([e]), np.array([0])) for e in error_range]
    ax.plot(error_range, losses, linewidth=2, label=f'q={q}')

ax.axvline(x=0, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('Error (y_pred - y_true)', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Quantile Loss\n(q=0.9 predicts 90th percentile)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# 3. Focal Loss vs BCE
ax = axes[1, 0]
y_pred_range = np.linspace(0.001, 0.999, 200)

bce = BinaryCrossEntropy()
bce_losses = [bce.forward(np.array([p]), np.array([1])) for p in y_pred_range]
ax.plot(y_pred_range, bce_losses, 'b-', linewidth=2, label='BCE')

for gamma in [0.5, 1, 2, 5]:
    focal = FocalLoss(gamma=gamma, alpha=0.5)
    focal_losses = [focal.forward(np.array([p]), np.array([1])) for p in y_pred_range]
    ax.plot(y_pred_range, focal_losses, linewidth=2, label=f'Focal (gamma={gamma})')

ax.set_xlabel('Predicted Probability', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Focal Loss vs BCE\n(True label = 1)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 4. Effect of Focal Loss on easy vs hard examples
ax = axes[1, 1]

# Compare loss for "easy" (correct with high confidence) vs "hard" (correct but low confidence) examples
easy_pred = 0.9  # Correct and confident
hard_pred = 0.6  # Correct but not confident

gammas = np.linspace(0, 5, 50)
easy_losses = []
hard_losses = []

for g in gammas:
    focal = FocalLoss(gamma=g, alpha=0.5)
    easy_losses.append(focal.forward(np.array([easy_pred]), np.array([1])))
    hard_losses.append(focal.forward(np.array([hard_pred]), np.array([1])))

ax.plot(gammas, easy_losses, 'g-', linewidth=2, label=f'Easy example (p={easy_pred})')
ax.plot(gammas, hard_losses, 'r-', linewidth=2, label=f'Hard example (p={hard_pred})')
ax.plot(gammas, np.array(hard_losses) / np.array(easy_losses), 'b--', linewidth=2, 
        label='Hard/Easy ratio')

ax.set_xlabel('Gamma', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Focal Loss: Down-weighting Easy Examples', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nFocal Loss Key Insight:")
print("As gamma increases, easy examples contribute less to the loss,")
print("allowing the model to focus on hard, misclassified examples.")

## 6. Gradient Flow Through Different Losses

Understanding how gradients flow through different losses is crucial for understanding training dynamics.

In [ ]:
# Visualize gradient flow for different losses

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Setup
error_range = np.linspace(-5, 5, 200)
y_pred_range = np.linspace(0.001, 0.999, 200)

# 1. MSE Gradient
ax = axes[0, 0]
mse = MeanSquaredError()
mse_grads = []
for e in error_range:
    mse.forward(np.array([e]), np.array([0]))
    mse_grads.append(mse.backward()[0])

ax.plot(error_range, mse_grads, 'b-', linewidth=2)
ax.fill_between(error_range, mse_grads, alpha=0.3)
ax.axhline(y=0, color='black', linestyle='--')
ax.axvline(x=0, color='black', linestyle='--')
ax.set_xlabel('Error', fontsize=12)
ax.set_ylabel('Gradient', fontsize=12)
ax.set_title('MSE Gradient\n(Linear in error)', fontsize=12)
ax.grid(True, alpha=0.3)

# 2. MAE Gradient
ax = axes[0, 1]
mae = MeanAbsoluteError()
mae_grads = []
for e in error_range:
    mae.forward(np.array([e]), np.array([0]))
    mae_grads.append(mae.backward()[0])

ax.plot(error_range, mae_grads, 'r-', linewidth=2)
ax.fill_between(error_range, mae_grads, alpha=0.3, color='red')
ax.axhline(y=0, color='black', linestyle='--')
ax.axvline(x=0, color='black', linestyle='--')
ax.set_xlabel('Error', fontsize=12)
ax.set_ylabel('Gradient', fontsize=12)
ax.set_title('MAE Gradient\n(Constant magnitude)', fontsize=12)
ax.grid(True, alpha=0.3)

# 3. Huber Gradient
ax = axes[0, 2]
huber = HuberLoss(delta=1.0)
huber_grads = []
for e in error_range:
    huber.forward(np.array([e]), np.array([0]))
    huber_grads.append(huber.backward()[0])

ax.plot(error_range, huber_grads, 'g-', linewidth=2)
ax.fill_between(error_range, huber_grads, alpha=0.3, color='green')
ax.axhline(y=0, color='black', linestyle='--')
ax.axvline(x=0, color='black', linestyle='--')
ax.axvline(x=1, color='gray', linestyle=':', alpha=0.5)
ax.axvline(x=-1, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Error', fontsize=12)
ax.set_ylabel('Gradient', fontsize=12)
ax.set_title('Huber Gradient\n(MSE inside delta, MAE outside)', fontsize=12)
ax.grid(True, alpha=0.3)

# 4. BCE Gradient (combined with sigmoid)
ax = axes[1, 0]

# When combined with sigmoid: grad = sigmoid(z) - y
z_range = np.linspace(-6, 6, 200)
y_true = 1
bce_combined_grad = sigmoid(z_range) - y_true

ax.plot(z_range, bce_combined_grad, 'purple', linewidth=2)
ax.fill_between(z_range, bce_combined_grad, alpha=0.3, color='purple')
ax.axhline(y=0, color='black', linestyle='--')
ax.axvline(x=0, color='black', linestyle='--')
ax.set_xlabel('Logit (z)', fontsize=12)
ax.set_ylabel('Gradient', fontsize=12)
ax.set_title('BCE + Sigmoid Gradient\n(Smooth, bounded)', fontsize=12)
ax.grid(True, alpha=0.3)

# 5. Gradient magnitude comparison
ax = axes[1, 1]

ax.semilogy(error_range[error_range != 0], np.abs(np.array(mse_grads)[error_range != 0]), 
            'b-', linewidth=2, label='MSE')
ax.semilogy(error_range[error_range != 0], np.abs(np.array(mae_grads)[error_range != 0]) + 0.001, 
            'r-', linewidth=2, label='MAE')
ax.semilogy(error_range[error_range != 0], np.abs(np.array(huber_grads)[error_range != 0]) + 0.001, 
            'g-', linewidth=2, label='Huber')

ax.set_xlabel('Error', fontsize=12)
ax.set_ylabel('|Gradient| (log scale)', fontsize=12)
ax.set_title('Gradient Magnitude Comparison', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# 6. Training dynamics simulation
ax = axes[1, 2]

def simulate_training(loss_class, y_true, initial_pred, lr=0.1, n_steps=50):
    """Simulate training with a given loss function."""
    loss_fn = loss_class()
    predictions = [initial_pred]
    pred = initial_pred
    
    for _ in range(n_steps):
        loss_fn.forward(np.array([pred]), np.array([y_true]))
        grad = loss_fn.backward()[0]
        pred = pred - lr * grad
        predictions.append(pred)
    
    return predictions

y_true = 5
initial = 0

mse_traj = simulate_training(MeanSquaredError, y_true, initial, lr=0.1)
mae_traj = simulate_training(MeanAbsoluteError, y_true, initial, lr=0.5)
huber_traj = simulate_training(HuberLoss, y_true, initial, lr=0.3)

ax.plot(mse_traj, 'b-', linewidth=2, label='MSE')
ax.plot(mae_traj, 'r-', linewidth=2, label='MAE')
ax.plot(huber_traj, 'g-', linewidth=2, label='Huber')
ax.axhline(y=y_true, color='black', linestyle='--', label='Target')

ax.set_xlabel('Step', fontsize=12)
ax.set_ylabel('Prediction', fontsize=12)
ax.set_title('Training Dynamics\n(Target = 5, Start = 0)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nGradient Flow Key Insights:")
print("1. MSE: Gradient proportional to error - fast initial progress, slows near optimum")
print("2. MAE: Constant gradient magnitude - steady progress but can oscillate at optimum")
print("3. Huber: Combines benefits - robust to outliers while smooth near optimum")

In [ ]:
# Demonstrate vanishing gradient problem

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Show why MSE with sigmoid has vanishing gradients
ax = axes[0]

z_range = np.linspace(-10, 10, 200)
y_true = 1

# BCE gradient (nice and stable)
bce_grad = sigmoid(z_range) - y_true

# MSE gradient (vanishes at saturation)
mse_grad = 2 * (sigmoid(z_range) - y_true) * sigmoid_derivative(z_range)

ax.plot(z_range, np.abs(bce_grad), 'b-', linewidth=2, label='|BCE gradient|')
ax.plot(z_range, np.abs(mse_grad), 'r-', linewidth=2, label='|MSE gradient|')

ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Logit (z)', fontsize=12)
ax.set_ylabel('|Gradient|', fontsize=12)
ax.set_title('Vanishing Gradient Problem\n(MSE + Sigmoid)', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Annotate
ax.annotate('MSE gradient\nvanishes here!', xy=(-8, 0.01), fontsize=10,
            bbox=dict(boxstyle='round', facecolor='red', alpha=0.3))

# Right: Training comparison
ax = axes[1]

def train_with_loss(loss_type, z_init=-5, lr=0.5, n_steps=100):
    """Simulate training a logistic regression model."""
    z = z_init
    history = [z]
    y_true = 1
    
    for _ in range(n_steps):
        p = sigmoid(z)
        
        if loss_type == 'BCE':
            # Gradient: dL/dz = p - y
            grad = p - y_true
        else:  # MSE
            # Gradient: dL/dz = 2(p - y) * p * (1 - p)
            grad = 2 * (p - y_true) * p * (1 - p)
        
        z = z - lr * grad
        history.append(z)
    
    return history

bce_history = train_with_loss('BCE', z_init=-5, lr=0.5)
mse_history = train_with_loss('MSE', z_init=-5, lr=0.5)

ax.plot(sigmoid(np.array(bce_history)), 'b-', linewidth=2, label='BCE')
ax.plot(sigmoid(np.array(mse_history)), 'r-', linewidth=2, label='MSE')
ax.axhline(y=1, color='green', linestyle='--', label='Target (y=1)')

ax.set_xlabel('Training Step', fontsize=12)
ax.set_ylabel('Predicted Probability', fontsize=12)
ax.set_title('Training from Wrong Initial Prediction\n(Starting with p=0.007)', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal predictions after 100 steps:")
print(f"BCE: {sigmoid(bce_history[-1]):.6f}")
print(f"MSE: {sigmoid(mse_history[-1]):.6f}")

---

## Interview Questions

### Question 1: Why do we use cross-entropy loss instead of MSE for classification?

**Answer:**

There are several reasons:

1. **Gradient properties**: When combined with sigmoid/softmax, cross-entropy gives gradients that are proportional to the error (prediction - target), while MSE gradients include the sigmoid derivative which vanishes for confident wrong predictions.

2. **Probabilistic interpretation**: Cross-entropy is derived from maximum likelihood estimation for Bernoulli/categorical distributions, making it the theoretically correct loss for classification.

3. **Faster convergence**: Cross-entropy provides larger gradients for wrong predictions, leading to faster learning.

4. **Convexity**: Cross-entropy is convex with respect to the logits when used with softmax, while MSE is not.

### Question 2: When would you use MAE over MSE for regression?

**Answer:**

Use MAE when:

1. **Outliers are present**: MAE is robust to outliers because it penalizes all errors equally, while MSE penalizes large errors heavily

2. **You want to predict the median**: MAE's optimal prediction is the median, while MSE's is the mean

3. **Errors have Laplace distribution**: MAE is the MLE for Laplace-distributed noise

Use MSE when:

1. **Large errors are particularly bad**: MSE penalizes large errors more

2. **Errors are Gaussian**: MSE is the MLE for Gaussian noise

3. **You need smooth gradients everywhere**: MSE is differentiable everywhere, MAE is not at zero

### Question 3: Explain Focal Loss and when you would use it.

**Answer:**

Focal Loss modifies cross-entropy by adding a focusing term:

$$FL = -\alpha (1-p_t)^\gamma \log(p_t)$$

where $p_t$ is the probability for the true class.

**How it works:**
- The $(1-p_t)^\gamma$ term down-weights the loss for well-classified examples
- As $\gamma$ increases, easy examples contribute less to the total loss
- This allows the model to focus on hard, misclassified examples

**When to use:**
1. **Class imbalance**: When positive examples are rare (e.g., 1:1000 ratio)
2. **Object detection**: Where background class dominates
3. **When easy negatives overwhelm the loss**: Standard cross-entropy can be dominated by easy examples, preventing learning of hard examples

---

## Exercises

### Exercise 1: Implement Log-Cosh Loss

Log-Cosh loss is another smooth alternative to MAE:

$$L = \sum_i \log(\cosh(y_i - \hat{y}_i))$$

It's approximately equal to MSE for small errors and MAE for large errors.

In [ ]:
# Exercise 1: Implement Log-Cosh Loss

class LogCoshLoss:
    """Log-Cosh Loss.
    
    L = sum(log(cosh(error)))
    
    TODO: Implement forward and backward methods
    Hint: d/dx log(cosh(x)) = tanh(x)
    """
    
    def __init__(self):
        self.cache = {}
    
    def forward(self, y_pred, y_true):
        # YOUR CODE HERE
        raise NotImplementedError("TODO: Implement this method")
    
    def backward(self):
        # YOUR CODE HERE
        raise NotImplementedError("TODO: Implement this method")

# Test your implementation
# Compare it with MSE, MAE, and Huber loss

### Exercise 2: Implement Dice Loss

Dice Loss is commonly used in image segmentation:

$$L = 1 - \frac{2|X \cap Y|}{|X| + |Y|} = 1 - \frac{2\sum p \cdot y}{\sum p + \sum y}$$

In [ ]:
# Exercise 2: Implement Dice Loss

class DiceLoss:
    """Dice Loss for segmentation tasks.
    
    L = 1 - (2 * intersection) / (sum_pred + sum_true)
    
    TODO: Implement forward and backward methods
    """
    
    def __init__(self, smooth=1e-6):
        self.smooth = smooth
        self.cache = {}
    
    def forward(self, y_pred, y_true):
        # YOUR CODE HERE
        raise NotImplementedError("TODO: Implement this method")
    
    def backward(self):
        # YOUR CODE HERE
        raise NotImplementedError("TODO: Implement this method")

### Exercise 3: Loss Function Comparison Study

Create a comprehensive study comparing different loss functions on a real regression task:
1. Generate or load a dataset with outliers
2. Train models with MSE, MAE, Huber, and your custom losses
3. Evaluate on clean test data and data with outliers
4. Visualize the predictions and compare

In [ ]:
# Exercise 3: Comprehensive loss function comparison

# Generate synthetic data with outliers
np.random.seed(42)
n_samples = 200
n_outliers = 20

# Clean data
X = np.random.uniform(-3, 3, n_samples).reshape(-1, 1)
y_clean = 2 * X.flatten() + 1 + np.random.normal(0, 0.5, n_samples)

# Add outliers
outlier_idx = np.random.choice(n_samples, n_outliers, replace=False)
y_with_outliers = y_clean.copy()
y_with_outliers[outlier_idx] += np.random.uniform(10, 20, n_outliers) * np.random.choice([-1, 1], n_outliers)

# YOUR CODE HERE:
# 1. Implement a simple linear regression model with configurable loss
# 2. Train with different losses
# 3. Compare and visualize results

---

## Summary

In this notebook, we covered:

1. **Cross-Entropy Loss**: The standard for classification, with nice gradient properties

2. **MSE vs MAE**: MSE penalizes large errors heavily (good for Gaussian noise), MAE is robust to outliers

3. **Loss Landscapes**: Visualizing how different losses shape the optimization problem

4. **Custom Losses**: Creating task-specific losses like asymmetric loss, quantile loss, and focal loss

5. **Gradient Flow**: Understanding how gradients behave and why some losses train better than others

### Key Takeaways

- Choose loss functions based on your task and data characteristics
- Cross-entropy is preferred for classification due to gradient properties
- Huber loss combines benefits of MSE and MAE for regression
- Custom losses can encode domain knowledge and improve performance
- Understanding gradient flow helps debug training issues